<a href="https://colab.research.google.com/github/CarlosJB95/PathIA-MSI-colon/blob/main/Notebook_Semana_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NOTEBOOK 1 - PYTHON INTERMEDIO**

---



# Notebook 1 — Python intermedio

**Proyecto:** Ruta IA + Patología Digital (PathIA 3.0) · Mes 1 · Semana 1

**Autor:** Carlos Jimenez B.

**Fecha:** Agosto 2026

**Estado:** Entregable de la Semana 1 (fundamentos de Python)

---

### Propósito
Notebook de aprendizaje que construye, desde cero, los fundamentos de Python
intermedio aplicados a patología computacional: de datos crudos de laminillas
a un manifiesto de cohorte con control de fuga a nivel paciente.

### Contenido
- **Funciones:** `*args`/`**kwargs`, closures, comprensión de listas.
- **Estructuras de datos:** diccionarios, sets (pacientes únicos), strings
  (parseo de barcode TCGA).
- **Pipeline procedural:** `extraer_patient_id`, `construir_manifest`, `hay_fuga`
  (control anti-fuga train/test a nivel paciente).
- **POO:** clases `Laminilla` (con QC vía `es_valida`) y `Manifiesto`
  (composición, resumen y anti-fuga), `__repr__`.
- **Archivos y excepciones:** escritura de CSV, `try/except`, `raise`.

### Entradas y salidas
- **Entrada:** Datos de ejemplo definidos en el propio notebook (identificadores
  tipo TCGA, conteos de parches, MPP). No requiere datos externos.
- **Salida:** `manifest.csv` (columnas: `slide_id`, `patient_id`, `n_parches`) —
  prototipo del manifiesto de cohorte del Mes 2.

### Cómo ejecutarlo
Entorno: Google Colab (Python 3). Ejecutar con
**Entorno de ejecución → Reiniciar sesión y ejecutar todo**. Corre entero,
de arriba a abajo, sin dependencias externas ni datos adicionales.

---
> ⚠️ **Material de aprendizaje — sin uso clínico.**

Este notebook es un ejercicio formativo. No es una herramienta diagnóstica ni está validado para decisiones clínicas. Los datos son ilustrativos, no provienen de casos reales de pacientes.

In [ ]:
#"Fija una semilla global (random.seed(42); np.random.seed(42))."
import random
import numpy as np

random.seed(42)
np.random.seed(42)

## 1. `*args` / `**kwargs`  

---



In [ ]:
def describir_parche(*coords, **metadatos):
    """Construye una descripción de un parche a partir de sus coordenadas y metadatos.

    Args:
        *coords: Coordenadas del parche pasadas como valores sueltos
            (p. ej. x, y, nivel).
        **metadatos: Pares clave-valor con información del parche. Se accede
            a las claves de forma directa (metadatos['tincion']), por lo que
            deben estar todas presentes: 'tincion', 'aumento', 'escaner', 'mpp'.

    Returns:
        str: Una línea legible que resume el parche.
    """
    return(f"Parche en {coords} · tincion {metadatos['tincion']} · aumento {metadatos['aumento']} · escaner {metadatos['escaner']} · MPP {metadatos['mpp']} µm/px")

In [ ]:
describir_parche(2048, 4096, tincion='H&E', aumento='20x', escaner='Aperio', mpp=0.5)

'Parche en (2048, 4096) · tincion H&E · aumento 20x · escaner Aperio · MPP 0.5 µm/px'

In [ ]:
def suma_nucleos (*cores):
    """Suma los conteos de núcleos de varios parches.

    Args:
        *cores: Conteos de núcleos pasados como valores sueltos (int).

    Returns:
        int: La suma total de núcleos.
    """
    return sum(cores)

In [ ]:
print(f'La suma de los núcleos es {suma_nucleos(40, 70, 80)}')

La suma de los núcleos es 190


In [ ]:
def meta_op(**metadatos):
    """Resume los metadatos de una WSI pasados por nombre, tolerando claves ausentes.

    Args:
        **metadatos: Pares clave-valor con metadatos de la WSI. Se acceden
            con .get(), que devuelve 'desconocido' si la clave falta.

    Returns:
        str: Resumen legible con escáner, aumento y MPP.
    """
    escaner = metadatos.get("escaner", 'desconocido')
    aumento = metadatos.get("aumento", 'desconocido')
    mpp     = metadatos.get("mpp", 'desconocido')
    return f'WSI · escaner {escaner} · aumento {aumento} · MPP {mpp}'

In [ ]:
print(meta_op(escaner="Aperio", aumento="40x", mpp=0.25))
print(meta_op(tincion="H&E"))

WSI · escaner Aperio · aumento 40x · MPP 0.25
WSI · escaner desconocido · aumento desconocido · MPP desconocido


In [ ]:
def resumen_laminilla (*args, **kwargs):
    """Resume una laminilla: nº de parches, % medio de tejido y escáner.

    Función de tipo AGREGADO (promedia sobre los parches), no filtro.
    Como divide entre el número de parches, no admite lote vacío
    (ZeroDivisionError si se llama sin parches).

    Args:
        *args: Porcentajes de tejido de los parches (int/float).
        **kwargs: Metadatos; requiere la clave 'escaner'.

    Returns:
        str: Resumen legible con conteo, % medio y escáner.
    """
    parches=len(args)
    tejprom=sum(args)/parches
    escaner=kwargs["escaner"]
    return(f'{parches} parches {tejprom}% de tejido promedio escaneados en "{escaner}"')

In [ ]:
resumen_laminilla(80, 60, 90, 70, escaner="Aperio", aumento="20x")

'4 parches 75.0% de tejido promedio escaneados en "Aperio"'

## **2. Funciones anidadas y Scope**

---



In [ ]:
def crear_filtro_tejido(umbral):
    """Fábrica de filtros de tejido: crea una función que decide si un parche pasa.

    Devuelve una función `es_tejido` que "recuerda" el `umbral` con el que se
    creó (closure). Permite generar filtros independientes con distintos
    umbrales (p. ej. uno al 30% y otro al 60%) sin interferir entre sí.

    Args:
        umbral (int): Porcentaje mínimo de tejido que exigirá el filtro creado.

    Returns:
        function: `es_tejido(pct)`, que devuelve True si `pct >= umbral`.
    """
    def es_tejido(pct):
        return pct >= umbral
    return es_tejido                  # devuelve la función interna (¡sin paréntesis!)

In [ ]:
#No funcional, solo para apartado "help()"
def es_tejido(pct):
    """Devuelve True si el porcentaje de tejido alcanza el umbral fijado."""
    return pct >= umbral

In [ ]:
filtro_30 = crear_filtro_tejido(30)

In [ ]:
filtro_30(85)

True

In [ ]:
filtro_30 = crear_filtro_tejido(30)
filtro_60 = crear_filtro_tejido(60)

In [ ]:
print(filtro_30(50))   #True
print(filtro_60(50))   #False

True
False


## **3. Comprension de listas**

---



In [ ]:
parches=[{'id':'p1', 'tejido_pct':80},
    {'id':'p2', 'tejido_pct':25}, #descartado
    {'id':'p3', 'tejido_pct':60}, #valido
    {'id':'p4', 'tejido_pct':95}, #valido
    {'id':'p5', 'tejido_pct':33}, #valido
    {'id':'p6', 'tejido_pct':16}] #descartado

In [ ]:
etiquetas=['valido' if p['tejido_pct']>30 else 'descartado' for p in parches]
print(etiquetas)

['valido', 'descartado', 'valido', 'valido', 'valido', 'descartado']


In [ ]:
validos=[p['id'] for p in parches if p['tejido_pct']>30]

In [ ]:
print(validos)

['p1', 'p3', 'p4', 'p5']


## ***MINI PROYECTO**

---



In [ ]:
def procesar_lote_parches(*parches, umbral_tejido=30, **opciones):
    """Filtra los parches de una laminilla por su porcentaje de tejido.
    Conserva únicamente los parches cuyo `tejido_pct` alcanza o supera
    `umbral_tejido`, y devuelve sus identificadores.

    Args:
        *parches: Parches sueltos, cada uno un dict con al menos las claves
            'id' (str) y 'tejido_pct' (0–100).
        umbral_tejido (int): Porcentaje mínimo de tejido exigido a cada
            parche para conservarlo. Por defecto 30.
        **opciones: Parámetros adicionales opcionales (reservados para uso
            futuro; actualmente no se utilizan).

    Returns:
        list[str]: Los 'id' de los parches que superan el umbral.
    """
    def pasa_filtro(p):
        return p['tejido_pct'] >= umbral_tejido
    resultado = [p['id'] for p in parches if pasa_filtro(p)]
    return resultado

In [ ]:
help(procesar_lote_parches)

Help on function procesar_lote_parches in module __main__:

procesar_lote_parches(*parches, umbral_tejido=30, **opciones)
    Filtra los parches de una laminilla por su porcentaje de tejido.
    Conserva únicamente los parches cuyo `tejido_pct` alcanza o supera
    `umbral_tejido`, y devuelve sus identificadores.

    Args:
        *parches: Parches sueltos, cada uno un dict con al menos las claves
            'id' (str) y 'tejido_pct' (0–100).
        umbral_tejido (int): Porcentaje mínimo de tejido exigido a cada
            parche para conservarlo. Por defecto 30.
        **opciones: Parámetros adicionales opcionales (reservados para uso
            futuro; actualmente no se utilizan).

    Returns:
        list[str]: Los 'id' de los parches que superan el umbral.



In [ ]:
procesar_lote_parches(
    {'id':'p1','tejido_pct':80},
    {'id':'p2','tejido_pct':25},
    {'id':'p3','tejido_pct':60},
    umbral_tejido=50
)

['p1', 'p3']

In [ ]:
# --- Casos límite de procesar_lote_parches (cierre Día 1) ---

# Lote de prueba: mezcla de tejido bueno, fondo casi vacío y borde exacto
lote_prueba = [
    {'id': 'parche_01', 'tejido_pct': 45},   # tejido claro
    {'id': 'fondo',     'tejido_pct': 2},    # casi todo cristal/fondo
    {'id': 'borde',     'tejido_pct': 0},    # frontera exacta
]
# Referencia — umbral normal (30): filtra de verdad
caso_normal = procesar_lote_parches(*lote_prueba, umbral_tejido=30)
print("Referencia (umbral=30):    ", caso_normal)

Referencia (umbral=30):     ['parche_01']


In [ ]:
# Caso 1 — cero parches: no hay nada que iterar → lista vacía
caso_cero = procesar_lote_parches(umbral_tejido=30)
print("Caso 1 (cero parches):     ", caso_cero)

Caso 1 (cero parches):      []


In [ ]:
# Caso 2 — umbral_tejido=0: 'tejido_pct >= 0' es True para todos → pasa todo
caso_umbral0 = procesar_lote_parches(*lote_prueba, umbral_tejido=0)
print("Caso 2 (umbral_tejido=0):  ", caso_umbral0)

Caso 2 (umbral_tejido=0):   ['parche_01', 'fondo', 'borde']


*“con lote vacío devuelve []; con umbral_tejido=0 no filtra nada (desactiva el control de tejido)”*

# **4. Diccionarios**

---



In [ ]:
slide_meta = {
    'TCGA-AA-3556': {'mpp': 0.5, 'escaner': 'Aperio', 'tincion': 'H&E', 'n_parches': 1200},
    'TCGA-AA-3712': {'mpp': 0.25, 'escaner': 'Hamamatsu', 'tincion': 'H&E', 'n_parches': 2048},
}

In [ ]:
slide_meta['TCGA-AA-3556'].get('gleason', 'no disponible')

'no disponible'

In [ ]:
slide_meta['TCGA-AA-3712'].get('escaner', 'desconocido')

'Hamamatsu'

In [ ]:
for x in slide_meta:
    print(x)

TCGA-AA-3556
TCGA-AA-3712


In [ ]:
for x in slide_meta:
  print(f"{x} → MPP {slide_meta[x]['mpp']}")

TCGA-AA-3556 → MPP 0.5
TCGA-AA-3712 → MPP 0.25


In [ ]:
for slide_id, meta in slide_meta.items():
    print(f"{slide_id} → MPP {meta['mpp']}")

TCGA-AA-3556 → MPP 0.5
TCGA-AA-3712 → MPP 0.25


In [ ]:
for slide_id, meta in slide_meta.items():
  print(f"{slide_id}: {meta['n_parches']} parches")
print(sum(meta['n_parches'] for meta in slide_meta.values()))

TCGA-AA-3556: 1200 parches
TCGA-AA-3712: 2048 parches
3248


In [ ]:
total = 0                                   # 1. ANTES del bucle: inicializa el acumulador
for meta in slide_meta.values():            # 2. recorre
    total = total + meta['n_parches']       # 3. DENTRO: acumula (total += ... es lo mismo)
print(total)

3248


# **5. Sets-Strings**

---



In [ ]:
# patient_id de cada laminilla de una cohorte (algunos pacientes aportan varias)
patient_ids = ['TCGA-AA-3556', 'TCGA-AA-3712', 'TCGA-AA-3556',
               'TCGA-AA-3901', 'TCGA-AA-3712', 'TCGA-AA-3556']

pacientes_unicos = set(patient_ids)

In [ ]:
pacientes_train = {'TCGA-AA-3556', 'TCGA-AA-3712'}
pacientes_test  = {'TCGA-AA-3901', 'TCGA-AA-3556'}   # ¡ojo con este split!

In [ ]:
pacientes_train & pacientes_test #'TCGA-AA-3556'
pacientes_train | pacientes_test #'TCGA-AA-3901', 'TCGA-AA-3712, TCGA-AA-3356
pacientes_train - pacientes_test #'TCGA-AA-3712'

{'TCGA-AA-3712'}

In [ ]:
def hay_fuga(pacientes_train, pacientes_test):
    """Indica si dos cohortes comparten algún paciente (fuga train/test).

    Args:
        pacientes_train (set): patient_id del conjunto de entrenamiento.
        pacientes_test (set): patient_id del conjunto de prueba.

    Returns:
        bool: True si comparten al menos un paciente (hay fuga),
            False si la intersección es vacía (split limpio).
    """
    return bool(pacientes_train & pacientes_test)

# Llamada a la función con los datos del notebook
resultado = hay_fuga(pacientes_train, pacientes_test)
print(resultado)

True


In [ ]:
# Split limpio: ningún paciente compartido
train_limpio = {'TCGA-AA-3556', 'TCGA-AA-3712'}
test_limpio  = {'TCGA-AA-3901', 'TCGA-AA-4001'}

print(hay_fuga(train_limpio, test_limpio))   # No, no hay datos que intersecten (en comun)

False


In [ ]:
slide_id = 'TCGA-AA-3556-01Z-00-DX1'
#           TCGA      AA      3556      01Z-00-DX1
#           └proyecto └centro └paciente └── muestra/porción/corte

In [ ]:
partes = slide_id.split('-')
print(partes[1])
print(partes[2])
print(partes[3])

AA
3556
01Z


In [ ]:
partes[0:3]   # → ['TCGA', 'AA', '3556']   ← sigue siendo una LISTA

['TCGA', 'AA', '3556']

In [ ]:
'-'.join(['TCGA', 'AA', '3556'])   # → 'TCGA-AA-3556'

'TCGA-AA-3556'

In [ ]:
patient_id = '-'.join(partes[0:3]) #['TCGA-AA-3556']

In [ ]:
## ANTES de validar: acepta basura en silencio → n_pacientes inflado (se corrige abajo con raise).
def extraer_patient_id(slide_id):
  partes=slide_id.split('-')
  return '-'.join(partes[0:3])

In [ ]:
print(extraer_patient_id('TCGA-AA-3556-01Z-00-DX1'))   # → TCGA-AA-3556
print(extraer_patient_id('TCGA-AA-3556-02Z-00-DX2'))   # → TCGA-AA-3556  (misma paciente, otra laminilla)
print(extraer_patient_id('TCGA-B0-4700-01Z-00-DX1'))   # → TCGA-B0-4700

TCGA-AA-3556
TCGA-AA-3556
TCGA-B0-4700


## ***construir-manifest** *v1*

In [ ]:
def construir_manifest(slides):
    """Construye el resumen (manifiesto) de una cohorte de laminillas.

    Recorre las laminillas, agrupa por paciente (vía extraer_patient_id) y
    acumula el total de parches.

    Args:
        slides (dict): Diccionario {slide_id: {'n_parches': int, ...}}.
            Cada clave es un barcode TCGA; su valor, los metadatos de esa
            laminilla.

    Returns:
        dict: Resumen de la cohorte con las claves:
            'n_slides' (int): número de laminillas.
            'n_pacientes' (int): número de pacientes únicos.
            'total_parches' (int): suma de parches de todas las laminillas.
            'pacientes' (set): conjunto de patient_id únicos (para hay_fuga).

    Raises:
        ValueError: Si algún slide_id no tiene el formato TCGA esperado
            (propagado desde extraer_patient_id).
    """
    pacientes = set()            # ← acumulador de pacientes únicos (set, colapsa duplicados)
    total_parches = 0            # ← acumulador de parches (patrón contador)
    for slide_id, meta in slides.items():
        pid = extraer_patient_id(slide_id)   # 'TCGA-AA-3556-01Z-00-DX1' → 'TCGA-AA-3556'
        pacientes.add(pid)                   # si ya está, el set lo ignora
        total_parches += meta['n_parches']   # acumula
    return {'n_slides':len(slides), 'n_pacientes':len(pacientes), 'total_parches':total_parches, 'pacientes': pacientes}

In [ ]:
slides = {
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},
    'TCGA-AA-3556-02Z-00-DX2': {'n_parches': 980},   # misma paciente, 2ª laminilla
    'TCGA-AA-3712-01Z-00-DX1': {'n_parches': 2048},
    'TCGA-B0-4700-01Z-00-DX1': {'n_parches': 1500},
}

In [ ]:
print(construir_manifest(slides))

{'n_slides': 4, 'n_pacientes': 3, 'total_parches': 5728, 'pacientes': {'TCGA-AA-3712', 'TCGA-B0-4700', 'TCGA-AA-3556'}}


In [ ]:
#Caso 1 Cohorte vacia
print(construir_manifest({}))

{'n_slides': 0, 'n_pacientes': 0, 'total_parches': 0, 'pacientes': set()}


In [ ]:
#Caso 2 Mal formado (ANTES de validar: acepta basura en silencio → n_pacientes inflado (se corrige abajo con raise).)
slides_raro = {
    'imagen_random.svs': {'n_parches': 500},   # ← no es un barcode TCGA
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},
}
print(construir_manifest(slides_raro))

{'n_slides': 2, 'n_pacientes': 2, 'total_parches': 1700, 'pacientes': {'imagen_random.svs', 'TCGA-AA-3556'}}


In [ ]:
def extraer_patient_id(slide_id):
    """Devuelve el patient_id (3 primeros campos) de un slide_id TCGA válido.

    Lanza ValueError si el slide_id no tiene el formato esperado.
    """
    partes = slide_id.split('-')
    if len(partes) < 3 or partes[0] != 'TCGA':
        raise ValueError(f"slide_id con formato inesperado: {slide_id!r}")
    return '-'.join(partes[:3])

In [ ]:
# Demostración: un slide_id mal formado lanza ValueError (capturado a propósito)
try:
    extraer_patient_id('imagen_random.svs')
except ValueError as e:
    print(f"✓ Rechazado correctamente → {e}")

✓ Rechazado correctamente → slide_id con formato inesperado: 'imagen_random.svs'


In [ ]:
#Caso 3 Slide duplicado --- Nada que arreglar (la estructura del diccionario lo arregla por default)
slides_dup = {
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},   # ← mismo slide_id, repetido
    'TCGA-AA-3712-01Z-00-DX1': {'n_parches': 2048},
}
print(construir_manifest(slides_dup))

{'n_slides': 2, 'n_pacientes': 2, 'total_parches': 3248, 'pacientes': {'TCGA-AA-3712', 'TCGA-AA-3556'}}


In [ ]:
#Caso 4 Solo UN slide/laminilla --- Nada que corregir
slides_solo = {
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},
    'TCGA-AA-3556-02Z-00-DX2': {'n_parches': 980},    # AA-3556: 2 laminillas
    'TCGA-B0-4700-01Z-00-DX1': {'n_parches': 1500},   # B0-4700: 1 sola laminilla
}
print(construir_manifest(slides_solo))

{'n_slides': 3, 'n_pacientes': 2, 'total_parches': 3680, 'pacientes': {'TCGA-B0-4700', 'TCGA-AA-3556'}}


## ****Leakage/comparar Cohortes**

In [ ]:
slides_train = {
    'TCGA-AA-3556-01Z-00-DX1': {'n_parches': 1200},
    'TCGA-AA-3712-01Z-00-DX1': {'n_parches': 2048},
}
slides_test = {
    'TCGA-AA-3556-02Z-00-DX2': {'n_parches': 980},    # ← ¡AA-3556 otra vez! (estaba en train)
    'TCGA-B0-4700-01Z-00-DX1': {'n_parches': 1500},
}

manifest_train = construir_manifest(slides_train)
manifest_test  = construir_manifest(slides_test)

fuga = hay_fuga(manifest_train['pacientes'], manifest_test['pacientes'])
print(f"¿Hay fuga entre train y test?: {fuga}")
#                └── set de train ──────────┘  └── set de test ──────────┘
#                        argumento 1              argumento 2

¿Hay fuga entre train y test?: True


# **6. Clases y Objetos**

---



In [ ]:
class Laminilla:
    """Objeto-expediente de una preparación (WSI): sus datos y su control de calidad.

    Atributos:
        slide_id (str): identificador/barcode de la laminilla.
        n_parches (int): número de parches extraídos.
        mpp (float): resolución del escaneo (µm/píxel).

    Métodos:
        patient_id(): extrae el patient_id de los 3 primeros campos del barcode.
        es_valida(): True si pasa el QC (MPP en 0.2–1.0 y campos no vacíos).
    """
    def __init__(self, slide_id, n_parches, mpp):
        self.slide_id = slide_id
        self.n_parches = n_parches
        self.mpp = mpp

    def patient_id(self):
        return'-'.join(self.slide_id.split('-')[:3])

    def es_valida(self):
      return bool(0.2<=self.mpp<=1.0 and self.slide_id)

    def __repr__(self):
      return f"Laminilla({self.slide_id}, {self.n_parches} parches, MPP {self.mpp} μm/px."

In [ ]:
lam_a = Laminilla('TCGA-AA-3556-01Z-00-DX1', 1200, 0.25)
lam_b = Laminilla('TCGA-B0-4700-01Z-00-DX1', 1500, 0.5)

In [ ]:
print(lam_a.slide_id)      # 1.'TCGA-AA-3556-01Z-00-DX1'
print(lam_a.n_parches)     # 2. 1200
print(lam_b.slide_id)      # 3. 'TCGA-B0-4700-01Z-00-DX1'

TCGA-AA-3556-01Z-00-DX1
1200
TCGA-B0-4700-01Z-00-DX1


In [ ]:
print(lam_a.patient_id())     # 'TCGA-AA-3556'

TCGA-AA-3556


In [ ]:
print(lam_a.es_valida())

True


In [ ]:
lam_ok  = Laminilla('TCGA-AA-3556-01Z-00-DX1', 1200, 0.5)   # MPP normal (20×)
lam_mal = Laminilla('TCGA-AA-3556-01Z-00-DX1', 1200, 1.8)   # MPP fuera de rango

print(lam_ok.es_valida())    # True
print(lam_mal.es_valida())   # False

True
False


In [ ]:
print(lam_ok)

Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 0.5 μm/px.


In [ ]:
lote = [lam_ok, lam_mal]
print(lote)                 #Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 0.5 μm/px.

[Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 0.5 μm/px., Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 1.8 μm/px.]


In [ ]:
for lam in lote:
    estado = "válida" if lam.es_valida() else "descartar"
    print(f"{lam}  →  {estado}")

Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 0.5 μm/px.  →  válida
Laminilla(TCGA-AA-3556-01Z-00-DX1, 1200 parches, MPP 1.8 μm/px.  →  descartar


In [ ]:
lam_vacia = Laminilla('', 1200, 0.5)   # MPP normal, pero slide_id VACÍO
print(lam_vacia.es_valida())           # False

False


In [ ]:
class Manifiesto:
    """Cohorte de laminillas: colección de Laminilla con resumen y control anti-fuga.

    Atributos:
        laminillas (list[Laminilla]): laminillas que componen la cohorte.

    Métodos:
        agregar(laminilla): añade una Laminilla a la cohorte.
        resumen(): dict con n_slides, n_pacientes, total_parches y el set de pacientes.
        hay_fuga(otro): True si comparte algún paciente con otro Manifiesto.
    """
    def __init__(self, laminillas=None):
      if laminillas is None:
        laminillas = []
      self.laminillas = laminillas       # guarda la lista de objetos Laminilla
    def resumen(self):
        pacientes = set()
        total_parches = 0
        for lam in self.laminillas:
            pacientes.add(lam.patient_id())
            total_parches += lam.n_parches
        return {'n_slides': len(self.laminillas), 'n_pacientes': len(pacientes), 'total_parches': total_parches, 'pacientes': pacientes}
    def hay_fuga(self, otro):
        return bool(self.resumen()['pacientes'] & otro.resumen()['pacientes'])
    def agregar(self, laminilla):
        self.laminillas.append(laminilla)      # añade una laminilla a la lista interna
    def a_csv(self, ruta):  # abre 'ruta' en modo escritura, escribe el encabezado,y una fila por cada laminilla en self.laminillas
      with open(ruta, 'w') as f:
        f.write('slide_id,patient_id,n_parches\n')          # encabezado
        for lam in self.laminillas:
          f.write(f'{lam.slide_id},{lam.patient_id()},{lam.n_parches}\n')
        #        └ atributo ──┘ └ método (con ()) ┘ └ atributo ─┘
def leer_manifest(ruta):
    try:
        with open(ruta, 'r') as f:
            print(f.read())
    except FileNotFoundError:
        print(f"No encontre el archivo en '{ruta}', ¿revisaste la ruta?")

In [ ]:
cohorte = Manifiesto([lam_a, lam_b])
print(cohorte.resumen())
# → {'n_slides': 2, 'n_pacientes': 2, 'total_parches': 2700, 'pacientes': {'TCGA-AA-3556', 'TCGA-B0-4700'}}

{'n_slides': 2, 'n_pacientes': 2, 'total_parches': 2700, 'pacientes': {'TCGA-B0-4700', 'TCGA-AA-3556'}}


In [ ]:
# Split sucio a propósito: lam_a (AA-3556) aparece en ambos
train = Manifiesto([lam_a, lam_b])
lam_c = Laminilla('TCGA-AA-3556-02Z-00-DX2', 980, 1.3)   # ← misma paciente AA-3556, otra laminilla
test  = Manifiesto([lam_c])

print(train.hay_fuga(test))    # True

True


In [ ]:
a = Manifiesto()
b = Manifiesto()
a.agregar(lam_a)
print(len(a.laminillas), len(b.laminillas))   # 2,3

1 0


In [ ]:
vacio = Manifiesto()
print(vacio.resumen())

{'n_slides': 0, 'n_pacientes': 0, 'total_parches': 0, 'pacientes': set()}


In [ ]:
cohorte = Manifiesto([lam_a, lam_b])
cohorte.a_csv('manifest.csv')

# leamos lo que se escribió para confirmar
with open('manifest.csv', 'r') as f:      # modo 'r' = leer
    print(f.read())

slide_id,patient_id,n_parches
TCGA-AA-3556-01Z-00-DX1,TCGA-AA-3556,1200
TCGA-B0-4700-01Z-00-DX1,TCGA-B0-4700,1500



In [ ]:
try:
    with open('no_existe.csv', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("No encontré el archivo; ¿revisaste la ruta?")

No encontré el archivo; ¿revisaste la ruta?


In [ ]:
leer_manifest('manifest.csv')     # 1. existe
leer_manifest('no_existe.csv')    # 2. no existe

slide_id,patient_id,n_parches
TCGA-AA-3556-01Z-00-DX1,TCGA-AA-3556,1200
TCGA-B0-4700-01Z-00-DX1,TCGA-B0-4700,1500

No encontre el archivo en 'no_existe.csv', ¿revisaste la ruta?


# **ANEXOS**

---



Términos de `procesar_lote_parches`

| Término | Tipo | Definición | Papel en la función | Ancla clínica |
|---|---|---|---|---|
| `*parches` | `tuple` de `dict` | Parches sueltos recogidos en una tupla; cada elemento es un parche. | Lo que se recorre y se filtra. | Los recortes (tiles) de una laminilla. |
| `umbral_tejido` | `int` (0–100), def. `30` | % mínimo de tejido exigido a cada parche. Uno solo para todo el lote. | Referencia contra la que se compara cada parche. | Criterio de calidad: "al menos X% de tejido". |
| `**opciones` | `dict` | Parámetros extra por nombre. Recibidos pero no usados en esta versión. | Reservado para ampliaciones futuras. | Espacio para ajustes futuros sin romper la firma. |
| `p` | `dict` | Variable del bucle: un parche individual por vuelta. | Cada parche que se evalúa. | La laminilla que miras en ese instante. |
| `p['id']` | `str` | Identificador único del parche. | Es lo que se extrae y devuelve. | Número de acceso / etiqueta del recorte. |
| `p['tejido_pct']` | `int`/`float` (0–100) | % de tejido de ese parche concreto. | Se compara contra `umbral_tejido`. | % de imagen que es tejido real, no fondo. |
| `pasa_filtro` | `function` anidada (closure) | Función interna True/False; recuerda `umbral_tejido`. | El criterio de decisión encapsulado. | La regla "¿tiene tejido suficiente para leerlo?". |
| `resultado` | `list[str]` | Lista de `id` de los parches que pasaron. | Lo que la función devuelve. | Lista de recortes aprobados para análisis. |

### Tabla de términos — `construir_manifest`

| Término | Tipo | Definición | Papel en la función | Ancla clínica |
|---|---|---|---|---|
| `slides` | dict de dicts | `{slide_id: {'n_parches': int, ...}}` | Lo que se recorre | La cohorte completa de laminillas |
| `slide_id` | str | Clave del dict; barcode de cada laminilla | Variable del bucle (clave) | Identificador/nombre de archivo de una WSI |
| `meta` | dict | Valor del dict; metadatos de esa laminilla | Variable del bucle (valor) | La ficha de datos de esa laminilla |
| `meta['n_parches']` | int | Nº de parches de esa laminilla | Se acumula | Recortes extraídos de esa WSI |
| `pid` | str | `patient_id` extraído del `slide_id` | Se agrega al set | El paciente dueño de esa laminilla |
| `pacientes` | set | Acumulador de `patient_id` únicos | Colapsa duplicados → pacientes únicos | Los pacientes distintos de la cohorte |
| `total_parches` | int | Acumulador (patrón contador) | Suma de parches | Total de recortes de la cohorte |
| `extraer_patient_id` | function | Saca el `patient_id` del barcode | Se llama por cada laminilla | Puente nombre-de-archivo → paciente |
| `n_slides` (retorno) | int | `len(slides)` | Clave del dict-resumen | Nº de laminillas |
| `n_pacientes` (retorno) | int | `len(pacientes)` | Clave del dict-resumen | Nº de pacientes únicos |
| `total_parches` (retorno) | int | Suma acumulada | Clave del dict-resumen | Total de parches |
| `pacientes` (retorno) | set | El set completo de `patient_id` | Insumo de `hay_fuga` | Materia prima del control anti-fuga |

### Tabla de términos — clase `Laminilla`

| Término | Tipo | Definición | Papel en la clase | Ancla clínica |
|---|---|---|---|---|
| `self` | referencia | El objeto concreto (esta laminilla) | Da acceso a sus propios datos | "Esta" laminilla en la platina |
| `slide_id` | atributo (str) | Identificador/barcode | Dato guardado en `__init__` | Nº de acceso de la WSI |
| `n_parches` | atributo (int) | Nº de parches | Dato guardado | Recortes extraídos |
| `mpp` | atributo (float) | Resolución (µm/píxel) | Dato guardado (base del QC) | Resolución del escaneo (40×≈0.25, 20×≈0.5) |
| `patient_id()` | método → str | Une los 3 primeros campos del barcode | Agrupa por paciente | Del nombre de archivo al paciente |
| `es_valida()` | método → bool | `True` si MPP∈[0.2, 1.0] y campos no vacíos | Control de calidad del objeto | Criterio de exclusión al microscopio |
| `__repr__()` | método → str | Representación legible del objeto | Impresión y depuración | Ver la laminilla de un vistazo |

### Tabla de términos — clase `Manifiesto`

| Término | Tipo | Definición | Papel en la clase | Ancla clínica |
|---|---|---|---|---|
| `self` | referencia | El objeto concreto (esta cohorte) | Da acceso a su estado | Esta cohorte de laminillas |
| `laminillas` | atributo (list[Laminilla]) | Lista de objetos `Laminilla` | Estado que se recorre | Las WSIs de la cohorte |
| `laminillas=None` | parámetro (centinela) | Default seguro; la lista `[]` se crea dentro | Evita el bug del default mutable | Cada cohorte nace vacía e independiente |
| `agregar(laminilla)` | método | `append` a `self.laminillas` | Hace crecer la cohorte | Añadir una WSI recién escaneada |
| `resumen()` | método → dict | `n_slides`, `n_pacientes`, `total_parches`, `pacientes` | Describe la cohorte | El resumen/manifiesto de la cohorte |
| `hay_fuga(otro)` | método → bool | Intersección de pacientes con otro `Manifiesto` | Control anti-fuga train/test | Detecta paciente compartido entre splits |
| `otro` | parámetro (Manifiesto) | El otro manifiesto a comparar | Operando de `hay_fuga` | El otro split (p. ej. test) |
| `a_csv(ruta)` | método | Escribe el manifiesto a un archivo CSV | Serializa el estado a disco | Produce `manifest.csv` persistente |
| `ruta` | parámetro (str) | Ruta/nombre del archivo de salida | Destino del CSV | Dónde se guarda el manifiesto |
| `lam` | var. de bucle (Laminilla) | Cada laminilla al recorrer `self.laminillas` | Iteración interna en `resumen`/`a_csv` | Cada WSI de la cohorte |

###     ***help()***

---





In [ ]:
help(procesar_lote_parches)

Help on function procesar_lote_parches in module __main__:

procesar_lote_parches(*parches, umbral_tejido=30, **opciones)
    Filtra los parches de una laminilla por su porcentaje de tejido.
    Conserva únicamente los parches cuyo `tejido_pct` alcanza o supera
    `umbral_tejido`, y devuelve sus identificadores.

    Args:
        *parches: Parches sueltos, cada uno un dict con al menos las claves
            'id' (str) y 'tejido_pct' (0–100).
        umbral_tejido (int): Porcentaje mínimo de tejido exigido a cada
            parche para conservarlo. Por defecto 30.
        **opciones: Parámetros adicionales opcionales (reservados para uso
            futuro; actualmente no se utilizan).

    Returns:
        list[str]: Los 'id' de los parches que superan el umbral.



In [ ]:
help(describir_parche)

Help on function describir_parche in module __main__:

describir_parche(*coords, **metadatos)
    Construye una descripción de un parche a partir de sus coordenadas y metadatos.

    Args:
        *coords: Coordenadas del parche pasadas como valores sueltos
            (p. ej. x, y, nivel).
        **metadatos: Pares clave-valor con información del parche. Se accede
            a las claves de forma directa (metadatos['tincion']), por lo que
            deben estar todas presentes: 'tincion', 'aumento', 'escaner', 'mpp'.

    Returns:
        str: Una línea legible que resume el parche.



In [ ]:
help(resumen_laminilla)

Help on function resumen_laminilla in module __main__:

resumen_laminilla(*args, **kwargs)
    Resume una laminilla: nº de parches, % medio de tejido y escáner.

    Función de tipo AGREGADO (promedia sobre los parches), no filtro.
    Como divide entre el número de parches, no admite lote vacío
    (ZeroDivisionError si se llama sin parches).

    Args:
        *args: Porcentajes de tejido de los parches (int/float).
        **kwargs: Metadatos; requiere la clave 'escaner'.

    Returns:
        str: Resumen legible con conteo, % medio y escáner.



In [ ]:
help(crear_filtro_tejido)

Help on function crear_filtro_tejido in module __main__:

crear_filtro_tejido(umbral)
    Fábrica de filtros de tejido: crea una función que decide si un parche pasa.

    Devuelve una función `es_tejido` que "recuerda" el `umbral` con el que se
    creó (closure). Permite generar filtros independientes con distintos
    umbrales (p. ej. uno al 30% y otro al 60%) sin interferir entre sí.

    Args:
        umbral (int): Porcentaje mínimo de tejido que exigirá el filtro creado.

    Returns:
        function: `es_tejido(pct)`, que devuelve True si `pct >= umbral`.



In [ ]:
help(es_tejido)

Help on function es_tejido in module __main__:

es_tejido(pct)
    Devuelve True si el porcentaje de tejido alcanza el umbral fijado.



In [ ]:
help(suma_nucleos)

Help on function suma_nucleos in module __main__:

suma_nucleos(*cores)
    Suma los conteos de núcleos de varios parches.

    Args:
        *cores: Conteos de núcleos pasados como valores sueltos (int).

    Returns:
        int: La suma total de núcleos.



In [ ]:
help(meta_op)

Help on function meta_op in module __main__:

meta_op(**metadatos)
    Resume los metadatos de una WSI pasados por nombre, tolerando claves ausentes.

    Args:
        **metadatos: Pares clave-valor con metadatos de la WSI. Se acceden
            con .get(), que devuelve 'desconocido' si la clave falta.

    Returns:
        str: Resumen legible con escáner, aumento y MPP.



In [ ]:
help(construir_manifest)

Help on function construir_manifest in module __main__:

construir_manifest(slides)
    Construye el resumen (manifiesto) de una cohorte de laminillas.

    Recorre las laminillas, agrupa por paciente (vía extraer_patient_id) y
    acumula el total de parches.

    Args:
        slides (dict): Diccionario {slide_id: {'n_parches': int, ...}}.
            Cada clave es un barcode TCGA; su valor, los metadatos de esa
            laminilla.

    Returns:
        dict: Resumen de la cohorte con las claves:
            'n_slides' (int): número de laminillas.
            'n_pacientes' (int): número de pacientes únicos.
            'total_parches' (int): suma de parches de todas las laminillas.
            'pacientes' (set): conjunto de patient_id únicos (para hay_fuga).

    Raises:
        ValueError: Si algún slide_id no tiene el formato TCGA esperado
            (propagado desde extraer_patient_id).



In [ ]:
help(extraer_patient_id)

Help on function extraer_patient_id in module __main__:

extraer_patient_id(slide_id)
    Devuelve el patient_id (3 primeros campos) de un slide_id TCGA válido.

    Lanza ValueError si el slide_id no tiene el formato esperado.



In [ ]:
help(hay_fuga)

Help on function hay_fuga in module __main__:

hay_fuga(pacientes_train, pacientes_test)
    Indica si dos cohortes comparten algún paciente (fuga train/test).

    Args:
        pacientes_train (set): patient_id del conjunto de entrenamiento.
        pacientes_test (set): patient_id del conjunto de prueba.

    Returns:
        bool: True si comparten al menos un paciente (hay fuga),
            False si la intersección es vacía (split limpio).



In [ ]:
help(Laminilla)

Help on class Laminilla in module __main__:

class Laminilla(builtins.object)
 |  Laminilla(slide_id, n_parches, mpp)
 |
 |  Objeto-expediente de una preparación (WSI): sus datos y su control de calidad.
 |
 |  Atributos:
 |      slide_id (str): identificador/barcode de la laminilla.
 |      n_parches (int): número de parches extraídos.
 |      mpp (float): resolución del escaneo (µm/píxel).
 |
 |  Métodos:
 |      patient_id(): extrae el patient_id de los 3 primeros campos del barcode.
 |      es_valida(): True si pasa el QC (MPP en 0.2–1.0 y campos no vacíos).
 |
 |  Methods defined here:
 |
 |  __init__(self, slide_id, n_parches, mpp)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(self)
 |      Return repr(self).
 |
 |  es_valida(self)
 |
 |  patient_id(self)
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref

In [ ]:
help(Manifiesto)

Help on class Manifiesto in module __main__:

class Manifiesto(builtins.object)
 |  Manifiesto(laminillas=None)
 |
 |  Cohorte de laminillas: colección de Laminilla con resumen y control anti-fuga.
 |
 |  Atributos:
 |      laminillas (list[Laminilla]): laminillas que componen la cohorte.
 |
 |  Métodos:
 |      agregar(laminilla): añade una Laminilla a la cohorte.
 |      resumen(): dict con n_slides, n_pacientes, total_parches y el set de pacientes.
 |      hay_fuga(otro): True si comparte algún paciente con otro Manifiesto.
 |
 |  Methods defined here:
 |
 |  __init__(self, laminillas=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  a_csv(self, ruta)
 |
 |  agregar(self, laminilla)
 |
 |  hay_fuga(self, otro)
 |
 |  resumen(self)
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references 